# transform()

## Definition
`transform()` applies a function to each group and returns a **result with the same shape and index as the original DataFrame**.
This is the key difference from `agg()`, which collapses groups.

### transform() vs agg()
| Feature | `agg()` | `transform()` |
|---------|---------|---------------|
| Returns | One row per group | Same rows as original |
| Use for | Summary stats | Broadcasting group stats back |
| Index | Group keys | Original index |
| Example | Group mean | Subtract group mean from each row |

### Common Use Cases in ML
- **Group-wise normalization** — subtract group mean / std
- **Lag features** — using groupby + shift + transform
- **Fill NaN with group mean** — `groupby().transform('mean')`
- **Min-max scale within group** — `groupby().transform(lambda x: (x-x.min())/(x.max()-x.min()))`

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'dept':   ['Eng', 'Mkt', 'Eng', 'HR', 'Eng', 'Mkt', 'HR', 'Eng'],
    'salary': [90000, 65000, 75000, 85000, 95000, 80000, 70000, 70000],
    'score':  [88, 72, 91, 75, 84, 78, 69, 82],
    'name':   ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank', 'Grace', 'Henry']
})
print(df)

In [ ]:
# transform with built-in function
# Broadcast group mean back to all rows
df['dept_mean_salary'] = df.groupby('dept')['salary'].transform('mean')
df['dept_mean_score']  = df.groupby('dept')['score'].transform('mean')

print('Group mean broadcast:')
print(df[['name', 'dept', 'salary', 'dept_mean_salary']].sort_values('dept'))

In [ ]:
# Z-score within each group (group-wise standardization)
df['salary_z'] = df.groupby('dept')['salary'].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)
print('Within-group Z-score:')
print(df[['name', 'dept', 'salary', 'salary_z']].round(3))

In [ ]:
# Fill NaN with group mean (group-aware imputation)
df_na = df.copy()
df_na.loc[[1, 3], 'salary'] = np.nan  # inject NaN

df_na['salary_filled'] = df_na.groupby('dept')['salary'].transform(
    lambda x: x.fillna(x.mean())
)
print('Group-mean imputation:')
print(df_na[['name', 'dept', 'salary', 'salary_filled']])

In [ ]:
# Group-wise min-max normalization
df['salary_minmax'] = df.groupby('dept')['salary'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8)
)
print('Min-max normalized within dept:')
print(df[['name', 'dept', 'salary', 'salary_minmax']].round(3))

In [ ]:
# transform with rank (rank within group)
df['dept_salary_rank'] = df.groupby('dept')['salary'].transform('rank', ascending=False)
print('Salary rank within dept:')
print(df[['name', 'dept', 'salary', 'dept_salary_rank']].sort_values(['dept', 'dept_salary_rank']))

In [ ]:
# transform to add group size
df['dept_size'] = df.groupby('dept')['name'].transform('count')
print('Group sizes added:')
print(df[['name', 'dept', 'dept_size']].drop_duplicates().sort_values('dept'))